In [1]:
"""
Build assumption_master sheet: multiple_loading
-----
Extracts the CTC % lookup table used by Spouse GLA / Spouse LSDB rows.
Range: Assumptions!$F$222:$G$224 (label → value).

Schema: category | label | value | notes
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib, csv, openpyxl, polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE_XLSX = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG = SNAPSHOT_DIR / "build_log.csv"

SHEET_NAME = "multiple_loading"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")


Building: multiple_loading


In [2]:
# Diagnostic: dump rows 218-230, cols F-H to see what's actually at $F$222:$G$224
print("Rows 218-230, cols E-I preview:")
print("-" * 90)
for row in range(218, 231):
    e = ws.cell(row=row, column=5).value  # E
    f = ws.cell(row=row, column=6).value  # F
    g = ws.cell(row=row, column=7).value  # G
    h = ws.cell(row=row, column=8).value  # H
    i = ws.cell(row=row, column=9).value  # I
    print(f"  Row {row:3d}: E={e!r:35s} F={f!r:35s} G={g!r:15s} H={h!r:15s} I={i!r}")

Rows 218-230, cols E-I preview:
------------------------------------------------------------------------------------------
  Row 218: E=None                                F='>65 & Retirement or Withdrawal only' G=0.02            H=None            I=None
  Row 219: E=None                                F=None                                G=None            H=None            I='Months of Disablement'
  Row 220: E=None                                F=None                                G=None            H=None            I='0 to 12'
  Row 221: E=None                                F='CTC'                               G=None            H=None            I='12 to 24'
  Row 222: E=None                                F='Synergy 1 yr payout'               G=0               H=None            I='24 to 36'
  Row 223: E=None                                F='Synergy 2 yr payout'               G=0               H=None            I='36 to 48'
  Row 224: E=None                                F='O

In [3]:
# =============================================================================
# CTC options: label → loading %
# Range: Assumptions rows 222-224, cols F (label) and G (value).
# Consumed by Spouse GLA row 22 and Spouse LSDB row 19 via
#   INDEX(Assumptions!$G$222:$G$224, MATCH(<CTC Definition>, $F$222:$F$224, 0), 1)
# =============================================================================

CTC_ROW_START = 222
CTC_ROW_END   = 225  # exclusive

all_rows: list[dict] = []

for row in range(CTC_ROW_START, CTC_ROW_END):
    label = ws.cell(row=row, column=6).value  # col F
    value = ws.cell(row=row, column=7).value  # col G
    
    if label is None:
        raise RuntimeError(f"Empty CTC label at row {row}")
    
    # Normalize label to snake_case option key
    option_key = str(label).lower().replace(" ", "_")
    
    all_rows.append({
        "option_key": option_key,
        "source_label": label,
        "loading": value,
        "source_row": row,
        "notes": "CTC premium loading. Consumed by Spouse GLA/LSDB CTC% lookup.",
    })

print(f"Total rows harvested: {len(all_rows)}")
print(f"\nAll rows preview:")
for r in all_rows:
    print(f"  {r['option_key']:25s} | source={r['source_label']!r:30s} | "
          f"loading={r['loading']} ({type(r['loading']).__name__})")

Total rows harvested: 3

All rows preview:
  synergy_1_yr_payout       | source='Synergy 1 yr payout'          | loading=0 (int)
  synergy_2_yr_payout       | source='Synergy 2 yr payout'          | loading=0 (int)
  other_disability          | source='Other disability'             | loading=0 (int)


In [4]:
def classify(v) -> str:
    if v is None:            return "null"
    if isinstance(v, bool):  return "bool"
    if isinstance(v, int):   return "int"
    if isinstance(v, float): return "float"
    return "string"

for r in all_rows:
    v = r["loading"]
    r["param_type"] = classify(v)
    r["loading_str"] = "" if v is None else str(v)

df = pl.DataFrame(all_rows).select([
    "option_key",
    pl.col("loading_str").alias("loading"),
    "param_type",
    "source_label",
    "source_row",
    "notes",
])

# Structural
assert df["option_key"].n_unique() == df.height, "Duplicate option_key"
assert df.height == 3, f"Expected 3 CTC options, got {df.height}"

# Expected keys
expected_keys = {"synergy_1_yr_payout", "synergy_2_yr_payout", "other_disability"}
actual_keys = set(df["option_key"].to_list())
assert actual_keys == expected_keys, (
    f"Key mismatch. Expected {expected_keys}, got {actual_keys}"
)

# Frozen values (all currently 0 — parameter slots)
for key in expected_keys:
    row = df.filter(pl.col("option_key") == key)
    val = float(row["loading"][0])
    assert val == 0.0, f"{key}: expected 0, got {val}"

print(f"✓ {df.height} CTC options validated")
print(f"✓ All 3 options currently 0 (parameter slots)")

print(f"\nType distribution:")
print(df.group_by("param_type").len().sort("param_type"))

print(f"\nFinal DataFrame:")
print(df)

✓ 3 CTC options validated
✓ All 3 options currently 0 (parameter slots)

Type distribution:
shape: (1, 2)
┌────────────┬─────┐
│ param_type ┆ len │
│ ---        ┆ --- │
│ str        ┆ u32 │
╞════════════╪═════╡
│ int        ┆ 3   │
└────────────┴─────┘

Final DataFrame:
shape: (3, 6)
┌────────────────────┬─────────┬────────────┬────────────────────┬────────────┬────────────────────┐
│ option_key         ┆ loading ┆ param_type ┆ source_label       ┆ source_row ┆ notes              │
│ ---                ┆ ---     ┆ ---        ┆ ---                ┆ ---        ┆ ---                │
│ str                ┆ str     ┆ str        ┆ str                ┆ i64        ┆ str                │
╞════════════════════╪═════════╪════════════╪════════════════════╪════════════╪════════════════════╡
│ synergy_1_yr_payou ┆ 0       ┆ int        ┆ Synergy 1 yr       ┆ 222        ┆ CTC premium        │
│ t                  ┆         ┆            ┆ payout             ┆            ┆ loading. Consumed  │
│       

In [5]:
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name}")
print(f"  Size:    {PARQUET_SIZE_KB:.2f} KB")
print(f"  SHA-256: {PARQUET_SHA[:16]}...")

master_wb = openpyxl.load_workbook(MASTER_XLSX)
if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated: {MASTER_XLSX.name}")

✓ Wrote multiple_loading.parquet
  Size:    2.88 KB
  SHA-256: 8565bc1d2c515989...
✓ Master workbook updated: assumption_master_A2026_07.xlsx
